In [1]:
import shap
import spacetimeformer as stf
import sys
sys.path.append('../../bats_transformer')
from data.bats_dataset import *
from tqdm import tqdm
import numpy as np
import pandas as pd

In [2]:
ignore_cols = ["FreqLedge","AmpK@end", "Fc", "FBak15dB  ", "FBak32dB", "EndF", "FBak20dB", "LowFreq", "Bndw20dB", 
               "CallsPerSec", "EndSlope", "SteepestSlope", "StartSlope", "Bndw15dB", "HiFtoUpprKnSlp", "HiFtoKnSlope", 
               "DominantSlope", "Bndw5dB", "PreFc500", "PreFc1000", "PreFc3000", "KneeToFcSlope", "TotalSlope", 
               "PreFc250", "CallDuration", "CummNmlzdSlp", "DurOf32dB", "SlopeAtFc", "LdgToFcSlp", "DurOf20dB", "DurOf15dB", 
               "TimeFromMaxToFc", "KnToFcDur", "HiFtoFcExpAmp", "AmpKurtosis", "LowestSlope", "KnToFcDmp", "HiFtoKnExpAmp", 
               "DurOf5dB", "KnToFcExpAmp", "RelPwr3rdTo1st", "LnExpB_StartAmp", "Filter", "HiFtoKnDmp", "LnExpB_EndAmp", 
               "HiFtoFcDmp", "AmpSkew", "LedgeDuration", "KneeToFcResidue", "PreFc3000Residue", "AmpGausR2", "PreFc1000Residue", 
               "Amp1stMean", "LdgToFcExp", "FcMinusEndF", "Amp4thMean", "HiFtoUpprKnExp", "HiFtoKnExp", "KnToFcExp", "UpprKnToKnExp", 
               "Kn-FcCurviness", "Amp2ndMean", "Quality", "HiFtoFcExp", "LnExpA_EndAmp", "RelPwr2ndTo1st", "LnExpA_StartAmp", 
               "HiFminusStartF", "Amp3rdMean", "PreFc500Residue", "Kn-FcCurvinessTrndSlp", "PreFc250Residue", "AmpVariance", "AmpMoment", 
               "meanKn-FcCurviness", "MinAccpQuality", "AmpEndLn60ExpC", "AmpStartLn60ExpC", "Preemphasis", "MaxSegLnght" ,"Max#CallsConsidered" ]
ignore_cols += ["Filename", "NextDirUp", 'Path', 'Version', 'Filter', 'Preemphasis', 'MaxSegLnght', "ParentDir", "file_id", "chirp_idx", "split"]

In [3]:
data_module = stf.data.DataModule(
    datasetCls = BatsCSVDataset,
    dataset_kwargs = {
        "root_path": "../../bats_transformer/data/july_daytime_chunked_quantile/splits",
        "prefix": "split",
        "ignore_cols": ignore_cols,
        "time_col_name": "TimeIndex",
        "val_split": 0.05,
        "test_split": 0.05,
        "context_points": None,
        "target_points": 1,
        "random_seed": 31
    },
    batch_size=64,
    workers=4
)

In [4]:
train_data = data_module.train_dataloader()
val_data = data_module.val_dataloader()
test_data = data_module.test_dataloader()

Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [5]:
means = np.zeros(32)
num_batches = 0
for batch in tqdm(train_data):
    x_t, x_c, y_t, y_c = batch
    # print(y_c.shape)
    # print(y_c.squeeze().mean(dim=0).numpy())
    means += y_c.squeeze().mean(dim=0).numpy()
    num_batches += (y_c.shape[0] / 64)
    # print(means)
means /= num_batches

100%|██████████| 389/389 [01:25<00:00,  4.52it/s]


In [6]:
means

array([ 6.16115427e-01,  8.37235193e-02,  1.11960428e-02,  3.04467208e-02,
        2.54538228e-02,  5.44536360e-03,  1.77633425e-02, -7.27580762e-04,
        2.40460329e-02, -1.38619238e-04, -2.41157765e-02, -2.40961849e-02,
       -1.57601660e-02, -1.94464227e+00, -6.87412494e-03,  6.08226954e-03,
        7.27094666e-03,  4.22954782e-04, -2.22743210e-03,  8.35934298e-03,
        1.02597064e-02,  2.80246194e-03,  7.33739550e-03, -1.75724779e-02,
        2.40103929e-02,  1.30930949e-02,  1.36398738e-02,  7.54809933e-03,
       -6.59676252e-02, -2.42164185e-02, -1.11347965e-01, -2.82066876e-03])

In [7]:
pred = means

In [8]:
truths = []
errors = []
preds = []

for batch in tqdm(test_data):
    x_t, x_c, y_t, y_c = batch
    for row in y_c:
        truths.append(row.numpy()[0])
        preds.append(pred)
        errors.append((row - pred).numpy()[0])

truths, preds, errors = np.array(truths), np.array(preds), np.array(errors)

100%|██████████| 22/22 [00:06<00:00,  3.34it/s]


In [9]:
pd.DataFrame(preds)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
1,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
2,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
3,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
4,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
1378,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
1379,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821
1380,0.616115,0.083724,0.011196,0.030447,0.025454,0.005445,0.017763,-0.000728,0.024046,-0.000139,...,0.007337,-0.017572,0.02401,0.013093,0.01364,0.007548,-0.065968,-0.024216,-0.111348,-0.002821


In [10]:
target_columns = train_data.dataset.target_cols
target_columns

['TimeInFile',
 'PrecedingIntrvl',
 'HiFreq',
 'Bndwdth',
 'FreqMaxPwr',
 'PrcntMaxAmpDur',
 'FreqKnee',
 'PrcntKneeDur',
 'StartF',
 'UpprKnFreq',
 'HiFtoUpprKnAmp',
 'HiFtoKnAmp',
 'HiFtoFcAmp',
 'UpprKnToKnAmp',
 'KnToFcAmp',
 'LdgToFcAmp',
 'FreqCtr',
 'FFwd32dB',
 'FFwd20dB',
 'FFwd15dB',
 'FBak5dB',
 'FFwd5dB',
 'Bndw32dB',
 'Amp1stQrtl',
 'Amp2ndQrtl',
 'Amp3rdQrtl',
 'Amp4thQrtl',
 '1st10kHzSlp',
 '1st5to15kHzSlp',
 '1st10kHzExp',
 '1st5to15kHzExp',
 'AmpK@start']

In [11]:
mae = np.abs(errors).mean(axis=0)
mse = (errors * errors).mean(axis=0)

In [12]:
mse

array([0.54205179, 0.81019935, 1.15981325, 1.18155539, 0.74730396,
       0.91815759, 0.84392829, 0.7675593 , 1.19829831, 0.99527604,
       0.99637977, 1.12647449, 0.95514742, 8.9790193 , 0.8830079 ,
       0.91137829, 0.8290068 , 0.99640506, 1.15394384, 1.12845471,
       0.71856137, 0.90229357, 0.70243054, 0.8260057 , 0.74087937,
       0.70200396, 0.92832896, 1.21863294, 1.47270401, 1.02081948,
       1.28453246, 0.84730643])

In [13]:
mse_df = pd.DataFrame(np.array([target_columns, mse]).T)
mse_df = mse_df.set_index(0)
# mse_df[1] = mse_df[1].round(6)
pd.Series(mse, index=target_columns)

TimeInFile         0.542052
PrecedingIntrvl    0.810199
HiFreq             1.159813
Bndwdth            1.181555
FreqMaxPwr         0.747304
PrcntMaxAmpDur     0.918158
FreqKnee           0.843928
PrcntKneeDur       0.767559
StartF             1.198298
UpprKnFreq         0.995276
HiFtoUpprKnAmp     0.996380
HiFtoKnAmp         1.126474
HiFtoFcAmp         0.955147
UpprKnToKnAmp      8.979019
KnToFcAmp          0.883008
LdgToFcAmp         0.911378
FreqCtr            0.829007
FFwd32dB           0.996405
FFwd20dB           1.153944
FFwd15dB           1.128455
FBak5dB            0.718561
FFwd5dB            0.902294
Bndw32dB           0.702431
Amp1stQrtl         0.826006
Amp2ndQrtl         0.740879
Amp3rdQrtl         0.702004
Amp4thQrtl         0.928329
1st10kHzSlp        1.218633
1st5to15kHzSlp     1.472704
1st10kHzExp        1.020819
1st5to15kHzExp     1.284532
AmpK@start         0.847306
dtype: float64

In [14]:
mse

array([0.54205179, 0.81019935, 1.15981325, 1.18155539, 0.74730396,
       0.91815759, 0.84392829, 0.7675593 , 1.19829831, 0.99527604,
       0.99637977, 1.12647449, 0.95514742, 8.9790193 , 0.8830079 ,
       0.91137829, 0.8290068 , 0.99640506, 1.15394384, 1.12845471,
       0.71856137, 0.90229357, 0.70243054, 0.8260057 , 0.74087937,
       0.70200396, 0.92832896, 1.21863294, 1.47270401, 1.02081948,
       1.28453246, 0.84730643])

In [15]:
errors.mean(axis=0)

array([-0.06037221,  0.18070645,  0.71640699,  0.56050445,  0.51269172,
        0.03821841,  0.45872898, -0.00808243,  0.72893276,  0.48029781,
        0.5351212 ,  0.45962028,  0.39541205,  0.30368116,  0.17720386,
        0.07441654,  0.56961543,  0.55275075,  0.69023605,  0.67686477,
        0.53619671,  0.59173783,  0.21898423, -0.07971172,  0.00144049,
       -0.05665032, -0.14419209,  0.59665741,  0.49897092,  0.510341  ,
        0.38109845, -0.03500851])

In [16]:
average_loss_per_row = mse_df.mean(axis=1)
average_loss_per_row_no_outlier = mse_df.drop("UpprKnToKnAmp", axis=0).mean(axis=1)
print(average_loss_per_row.mean(), average_loss_per_row_no_outlier.mean())

1.2027456132167464 0.9518980747722591
